In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target


X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, :-1], df.iloc[:, -1], test_size=0.25, random_state=42)

X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

X_train = torch.FloatTensor(X_train.values)
X_valid = torch.FloatTensor(X_valid.values)
X_test = torch.FloatTensor(X_test.values)
means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdim=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

y_train = torch.FloatTensor(y_train.values).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid.values).reshape(-1, 1)
y_test = torch.FloatTensor(y_test.values).reshape(-1, 1)

In [2]:
deep_feature_num:int = X_train.shape[1]

device = "cuda"

In [14]:
class WideAndDeep(nn.Module):
    def __init__(self, n_features):
        super(WideAndDeep, self).__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features, 1)

    def forward(self, X):
        deep_output = self.deep_stack(X)
        wide_and_deep = torch.concat([X,deep_output], dim=1)
        return self.output_layer(wide_and_deep)

class WideAndDeepDataset(torch.utils.data.Dataset):
    def __init__(self, X_wide, X_deep, y):
        self.X_wide = X_wide
        self.X_deep = X_deep
        self.y = y
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        input_dict = {"X_wide": self.X_wide[idx], "X_deep":self.X_deep[idx]}
        return input_dict, self.y[idx]

In [6]:
n_features = X_train.shape[1]
torch.manual_seed(42)
model = WideAndDeep(n_features).to(device)
learning_rate = 0.002

In [7]:
class WideAndDeepV2(nn.Module):
    def __init__(self, n_features):
        super(WideAndDeep, self).__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torchmetrics

train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)

In [11]:
def training(model, optimizer, criterion, dataloader:DataLoader, n_epoch:int):
    model.train()
    optimizer.zero_grad()
    for i in range(n_epoch):
        for X_batch_wide, X_batch_deep, y_batch in dataloader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)
            
            y_pred = model(X_batch_wide, X_batch_deep)
            loss = criterion(y_pred, y_batch)
            optimizer.step()
            optimizer.zero_grad()
            print(f"Epoch {i + 1}/{n_epoch}, Loss: {loss.item()}")


def evaluating(model, dataloader:DataLoader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch_wide, X_batch_deep, y_batch in dataloader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)
            
            y_pred = model(X_batch_wide, X_batch_deep)
            metric.update(y_pred, y_batch)
    
    return metric.compute()

In [15]:
train_data_named = WideAndDeepDataset(X_wide = X_train[:, :5], X_deep=X_train[:, 2:], y=y_train)
train_loader_named = DataLoader(train_data_named, batch_size=32, shuffle=True)

In [ ]:
#didn't test these codes
class WideAndDeepV4(nn.Module):
    def __init__(self, n_features):
        super(WideAndDeep, self).__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features, 1)
        self.aux_output_layer = nn.Linear(40,1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        main_output = self.output_layer(wide_and_deep)
        aux_output = self.aux_output_layer(deep_output)
        
        return main_output, aux_output
    
def training_V4(model, optimizer, criterion, dataloader:DataLoader, n_epoch:int):
    model.train()
    optimizer.zero_grad()
    for i in range(n_epoch):
        for inputs, y_batch in dataloader:
            inputs = [input.to(device) for input in inputs]
            y_batch = y_batch.to(device)
            y_pred, y_pred_aux = model(*inputs)
            main_loss = criterion(y_pred, y_batch)
            aux_loss = criterion(y_pred_aux, y_batch)
            loss = 0.8 * main_loss + 0.2 * aux_loss
            optimizer.step()
            optimizer.zero_grad()
            print(f"Epoch {i + 1}/{n_epoch}, Loss: {loss.item()}")


def evaluating_V4(model, dataloader:DataLoader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for inputs, y_batch in dataloader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)
            
            y_pred = model(X_batch_wide, X_batch_deep)
            metric.update(y_pred, y_batch)
    
    return metric.compute()